In [1]:
import torch

In [2]:
# 1. Data gathering
# 2. Data preprocessing
# 3. Feature engineering
# 4. Model training
# 5. Testing

In [3]:
## Dataset used

# https://www.kaggle.com/datasets/mirichoi0218/insurance

In [4]:
!pip install kaggle

In [5]:
import kagglehub

# Download latest version

path = kagglehub.dataset_download("mirichoi0218/insurance")
print(f"Path of dataset files: {path}")

Using Colab cache for faster access to the 'insurance' dataset.
Path of dataset files: /kaggle/input/insurance


In [6]:
import os
print(os.getcwd())

/content


In [7]:
import pandas as pd


In [8]:
os.listdir('/kaggle/input/insurance')

['insurance.csv']

In [9]:
df = pd.read_csv(os.path.join(path, 'insurance.csv'))

In [10]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [12]:
df.describe()

,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


In [13]:
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [14]:
# Split dataset before encoding

train_df, test_df = train_test_split(df, test_size = 0.2, random_state = 42)

In [15]:
# Encode categorical variable

label_encoder = {}
for col in ['sex', 'smoker', 'region']:
  le = LabelEncoder()
  train_df[col] = le.fit_transform(train_df[col])
  test_df[col] = le.transform(test_df[col])
  label_encoder[col] = le


In [16]:
# Features and target

X_train = train_df.drop(columns = ["charges"])
Y_train = train_df["charges"]
X_test = test_df.drop(columns = ["charges"])
Y_test = test_df["charges"]

In [17]:
print(X_train.head())

      age  sex    bmi  children  smoker  region
560    46    0  19.95         2       0       1
1285   47    0  24.32         0       0       0
1142   52    0  24.86         0       0       2
969    39    0  34.32         5       0       2
486    54    0  21.47         3       0       1


In [18]:
print(X_test.head())

      age  sex     bmi  children  smoker  region
764    45    0  25.175         2       0       0
887    36    0  30.020         0       0       1
890    64    0  26.885         0       1       1
1293   46    1  25.745         3       0       1
259    19    1  31.920         0       1       1


In [19]:
# Normalize features

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [20]:
# Print normalized training features array
print("Normalized X_train:")
print(X_train[:5])
print(f"\nShape: {X_train.shape}")

Normalized X_train:
[[ 0.47222651 -1.0246016  -1.75652513  0.73433626 -0.50874702 -0.45611589]
 [ 0.54331294 -1.0246016  -1.03308239 -0.91119211 -0.50874702 -1.35325561]
 [ 0.8987451  -1.0246016  -0.94368672 -0.91119211 -0.50874702  0.44102382]
 [-0.02537852 -1.0246016   0.62239299  3.20262882 -0.50874702  0.44102382]
 [ 1.04091797 -1.0246016  -1.50489287  1.55710045 -0.50874702 -0.45611589]]

Shape: (1070, 6)


In [21]:
# Convert to tensors

X_train_tensor = torch.tensor(X_train, dtype = torch.float32)
Y_train_tensor = torch.tensor(Y_train.values, dtype = torch.float32).view(-1,1)
X_test_tensor = torch.tensor(X_test, dtype = torch.float32)
Y_test_tensor = torch.tensor(Y_test.values, dtype = torch.float32).view(-1,1)


In [22]:
print(Y_train_tensor)
print(Y_train_tensor.shape)

tensor([[ 9193.8389],
        [ 8534.6719],
        [27117.9941],
        ...,
        [11931.1250],
        [46113.5117],
        [10214.6357]])
torch.Size([1070, 1])


In [23]:
# Define Neural Network

class SimpleNNRegressionModel(nn.Module):
  def __init__(self, input_dim):
    super(SimpleNNRegressionModel, self).__init__()
    self.network = nn.Sequential(
        nn.Linear(input_dim, 64),
        nn.ReLU(),
        nn.Linear(64,128),
        nn.ReLU(),
        nn.Linear(128, 1)
    )

  def forward(self,x):
    return self.network(x)

In [24]:
X_train_tensor.shape

torch.Size([1070, 6])

In [25]:
input_dim = X_train_tensor.shape[1]
model = SimpleNNRegressionModel(input_dim)

In [26]:
print(model)

SimpleNNRegressionModel(
  (network): Sequential(
    (0): Linear(in_features=6, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=1, bias=True)
  )
)


In [27]:
# Loss and Optimizer

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr = 0.01)


In [28]:
# Training Loop
epochs = 10000

for epoch in range(epochs):
  model.train()
  optimizer.zero_grad()
  predictions = model(X_train_tensor)
  loss = criterion(predictions, Y_train_tensor)
  loss.backward()

  optimizer.step()

  if (epoch + 1) % 100 == 0:
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [100/10000], Loss: 44799356.0000
Epoch [200/10000], Loss: 31318246.0000
Epoch [300/10000], Loss: 28441818.0000
Epoch [400/10000], Loss: 26867218.0000
Epoch [500/10000], Loss: 25953024.0000
Epoch [600/10000], Loss: 25354654.0000
Epoch [700/10000], Loss: 24910990.0000
Epoch [800/10000], Loss: 24573392.0000
Epoch [900/10000], Loss: 24238728.0000
Epoch [1000/10000], Loss: 23836608.0000
Epoch [1100/10000], Loss: 23491586.0000
Epoch [1200/10000], Loss: 23185548.0000
Epoch [1300/10000], Loss: 22929928.0000
Epoch [1400/10000], Loss: 22724306.0000
Epoch [1500/10000], Loss: 22566522.0000
Epoch [1600/10000], Loss: 22390888.0000
Epoch [1700/10000], Loss: 22216904.0000
Epoch [1800/10000], Loss: 21943184.0000
Epoch [1900/10000], Loss: 21612606.0000
Epoch [2000/10000], Loss: 21375348.0000
Epoch [2100/10000], Loss: 21132936.0000
Epoch [2200/10000], Loss: 20744204.0000
Epoch [2300/10000], Loss: 20531248.0000
Epoch [2400/10000], Loss: 20315604.0000
Epoch [2500/10000], Loss: 20126738.0000
Epoch [26

In [29]:
# Model Evaluation

model.eval()
Y_pred = model(X_test_tensor).detach().numpy()


In [30]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error



Y_test_numpy = Y_test_tensor.numpy()


# Calculate metrics


mse = mean_squared_error(Y_test_numpy, Y_pred)
r2 = r2_score(Y_test_numpy, Y_pred)
rmse = mse ** 0.5
mae = mean_absolute_error(Y_test_numpy, Y_pred)

# Print the metrics
print(f"Mean Squared Error (MSE): {mse}")
print(f"Root Mean Squared Error (RMSE): {rmse}")
print(f"R-squared (R2) Score: {r2}")
print(f"Mean Absolute Error (MAE): {mae}")

Mean Squared Error (MSE): 34877424.0
Root Mean Squared Error (RMSE): 5905.7111341480295
R-squared (R2) Score: 0.7753448486328125
Mean Absolute Error (MAE): 3737.3232421875


In [31]:
def predict_charges(age, sex, bmi, children, smoker, region):
  # Assign the created DataFrame to input_data
  input_data = pd.DataFrame([[age, sex, bmi, children, smoker, region]],
                            columns = ["age", "sex", "bmi", "children", "smoker", "region"])

  # First encode all categorical columns
  for col in ["sex", "smoker", "region"]:
    input_data[col] = label_encoder[col].transform(input_data[col])

  # Scale the complete input data, convert to tensor, and predict outside the loop
  input_data_scaled = scaler.transform(input_data)
  input_data_tensor = torch.tensor(input_data_scaled, dtype = torch.float32)

  model.eval()
  with torch.no_grad():
    predicted_charge = model(input_data_tensor).item()
  return predicted_charge

In [32]:
predicted = predict_charges(19, 'female', 27.9, 0, 'yes', 'southwest')

In [33]:
# Print the predicted charge
print(f"Predicted insurance charge: ${predicted:,.2f}")

Predicted insurance charge: $19,134.93


1. Dataset(torch.utils.data.Dataset)
2. DataLoader(torch.utils.data.DataLoader)

In [34]:
# Creating our custom Dataset in Pytorch
# init() - initialised the dataset, load data, applied preprocessing
# len() - return the total numbers of samples in the dataset
# getitem() - Defines how to retrieve a single data sample when an index is provided

In [35]:
import torch
from torch.utils.data import Dataset, DataLoader

In [39]:
class InsuranceDataset(Dataset):
  def __init__(self, X, Y):
    self.X = X
    self.Y = Y

  def __len__(self):
      return len(self.X)

  def __getitem__(self, idx):
      features = torch.tensor(self.X[idx], dtype = torch.float32)
      target = torch.tensor(self.Y[idx], dtype = torch.float32)
      return features, target



In [44]:
dataset = InsuranceDataset(X_train, Y_train.values)

In [45]:
dataloader= DataLoader(dataset, batch_size = 32, shuffle = True)

In [47]:
for batch_idx, (features, targets) in enumerate(dataloader):
  print(f"Batch {batch_idx+1} :")
  print(f"Features: {features.shape}")
  print(f"Targets: {targets.shape}")
  break

Batch 1 :
Features: torch.Size([32, 6])
Targets: torch.Size([32])


In [ ]:
 # 1000 -> total number of data
 # batch-size -> 100
 # epoch -> 100

In [48]:
epochs = 10000

for epoch in range(epochs):
  model.train()
  for batch_idx,(batch_X, batch_Y) in enumerate(dataloader):

    print(f"Current batch: {batch_idx}")
    optimizer.zero_grad()
    predictions = model(X_train_tensor)
    loss = criterion(predictions, Y_train_tensor)
    loss.backward()
    optimizer.step()
    print(f"Batch [{batch_idx+1}/{epochs}], Loss: {loss.item():.4f}")

  if (epoch + 1) % 100 == 0:
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

Streaming output truncated to the last 5000 lines.
Batch [17/10000], Loss: 2164747.2500
Current batch: 17
Batch [18/10000], Loss: 2325191.2500
Current batch: 18
Batch [19/10000], Loss: 2518674.0000
Current batch: 19
Batch [20/10000], Loss: 2514345.7500
Current batch: 20
Batch [21/10000], Loss: 2416757.5000
Current batch: 21
Batch [22/10000], Loss: 2092697.6250
Current batch: 22
Batch [23/10000], Loss: 1723355.2500
Current batch: 23
Batch [24/10000], Loss: 1413382.7500
Current batch: 24
Batch [25/10000], Loss: 1291867.0000
Current batch: 25
Batch [26/10000], Loss: 1365032.5000
Current batch: 26
Batch [27/10000], Loss: 1541715.5000
Current batch: 27
Batch [28/10000], Loss: 1704961.6250
Current batch: 28
Batch [29/10000], Loss: 1750809.1250
Current batch: 29
Batch [30/10000], Loss: 1683039.2500
Current batch: 30
Batch [31/10000], Loss: 1522818.6250
Current batch: 31
Batch [32/10000], Loss: 1370795.8750
Current batch: 32
Batch [33/10000], Loss: 1293363.0000
Current batch: 33
Batch [34/1000